# Programowanie w języku Python

## Wykład 5 — Funkcje II: elastyczne interfejsy i funkcje jako obiekty

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć rozwinięcie argumentów przez `*` i `**`,
- umieć korzystać z `*args` i `**kwargs`,
- rozumieć argumenty positional-only i keyword-only,
- wiedzieć, że funkcja w Pythonie jest obiektem,
- umieć przekazać funkcję jako argument i zwrócić ją z funkcji,
- rozumieć podstawowe zastosowanie `lambda`,
- znać ideę funkcji zagnieżdżonej, closure i `nonlocal`,
- widzieć związek tych mechanizmów z późniejszymi dekoratorami.

## Powtórzenie z Wykładu 4

Potrafimy już napisać:

```python
def is_alarm(temperature, limit=30):
    return temperature > limit
```

i wywołać:

```python
is_alarm(35)
is_alarm(35, 40)
is_alarm(temperature=35, limit=40)
```

Dzisiaj rozszerzymy ten model o bardziej elastyczne interfejsy funkcji.

## Najpierw: co robi `*` podczas wywołania?

Mamy listę:

```python
numbers = [10, 20, 30]
```

Porównaj:

```python
print(numbers)
print(*numbers)
```

W drugim przypadku elementy listy są rozwijane do osobnych argumentów.

In [ ]:
numbers = [10, 20, 30]

print(numbers)
print(*numbers)

## Rozwijanie argumentów pozycyjnych

Załóżmy funkcję:

```python
def add_three(a, b, c):
    return a + b + c
```

Możemy wywołać ją klasycznie:

```python
add_three(1, 2, 3)
```

albo rozwinąć sekwencję:

```python
values = [1, 2, 3]
add_three(*values)
```

In [ ]:
def add_three(a, b, c):
    return a + b + c

values = [1, 2, 3]

print(add_three(1, 2, 3))
print(add_three(*values))

## Co oznacza `*`?

W wywołaniu:

```python
function(*values)
```

`*` mówi w uproszczeniu:

> potraktuj kolejne elementy obiektu iterowalnego jako osobne argumenty pozycyjne.

To przydaje się nie tylko dla list, ale również np. dla krotek.

In [ ]:
point = (10, 20)

def show_point(x, y):
    print(f"x={x}, y={y}")

show_point(*point)

## Rozwijanie argumentów nazwanych przez `**`

Mamy słownik:

```python
student = {
    "name": "Anna",
    "age": 20
}
```

Jeżeli klucze odpowiadają nazwom parametrów, możemy napisać:

```python
describe_student(**student)
```

In [ ]:
def describe_student(name, age):
    print(f"{name} ma {age} lat")

student = {
    "name": "Anna",
    "age": 20
}

describe_student(**student)

## `*` i `**` przy wywołaniu

W uproszczeniu:

```text
*obiekt
↓
rozwija argumenty pozycyjne

**słownik
↓
rozwija argumenty nazwane
```

To jest pierwszy kierunek działania tych operatorów.

## Drugi kierunek: zbieranie argumentów

Możemy odwrócić sytuację.

Jeżeli nie wiemy, ile argumentów pozycyjnych otrzyma funkcja:

```python
def function(*args):
    ...
```

Python zbierze je do krotki.

In [ ]:
def show_arguments(*args):
    print(args)
    print(type(args))

show_arguments(10, 20, 30)
show_arguments("AGH", 3.14, True)

## `*args`

Nazwa `args` jest konwencją, nie specjalnym słowem języka.

Możliwe byłoby:

```python
def function(*values):
    ...
```

ale najczęściej spotkasz:

```python
def function(*args):
    ...
```

Najważniejsza jest **gwiazdka**.

## Praktyczny przykład `*args`

Funkcja może obliczać średnią z dowolnej liczby argumentów:

In [ ]:
def average(*values):
    if not values:
        return None

    return sum(values) / len(values)

print(average(10, 20))
print(average(10, 20, 30, 40))
print(average())

## `**kwargs`

Analogicznie:

```python
def function(**kwargs):
    ...
```

zbiera dodatkowe argumenty nazwane do słownika.

In [ ]:
def show_options(**kwargs):
    print(kwargs)
    print(type(kwargs))

show_options(timeout=10, retries=3, verbose=True)

## Praktyczny przykład `**kwargs`

Możemy przyjąć zestaw opcjonalnych ustawień:

In [ ]:
def configure_sensor(name, **options):
    print("Sensor:", name)

    for key, value in options.items():
        print(f"  {key} = {value}")

configure_sensor(
    "temperature_1",
    unit="C",
    frequency=10,
    active=True
)

## Najważniejsza symetria

```text
WYWOŁANIE                  DEFINICJA

*values                    *args
rozwija sekwencję          zbiera argumenty
na argumenty               do tuple

**options                  **kwargs
rozwija słownik            zbiera argumenty
na argumenty nazwane       do dict
```

To lepszy model niż uczenie się `*args` i `**kwargs` jako dwóch niezależnych sztuczek.

## Możemy je łączyć

Przykładowa funkcja:

```python
def report(title, *values, **options):
    ...
```

ma:

- jeden zwykły parametr `title`,
- dowolną liczbę dodatkowych argumentów pozycyjnych,
- dowolną liczbę dodatkowych argumentów nazwanych.

In [ ]:
def report(title, *values, **options):
    print("Tytuł:", title)
    print("Wartości:", values)
    print("Opcje:", options)

report(
    "Temperatury",
    21.5, 22.1, 23.0,
    unit="C",
    precision=1
)

## Elastyczność ma koszt

Funkcja:

```python
def report(title, *args, **kwargs):
    ...
```

jest bardzo elastyczna, ale mniej jasno określa swój interfejs.

Nie należy używać `*args` i `**kwargs` tylko dlatego, że istnieją.

> Im bardziej precyzyjna sygnatura funkcji, tym łatwiej zrozumieć jej użycie.

## Argumenty keyword-only

Czasem chcemy wymusić podawanie wybranych argumentów po nazwie.

Służy do tego `*` w definicji:

In [ ]:
def configure(device, *, timeout=10, retries=3):
    print(device, timeout, retries)

configure("sensor")
configure("sensor", timeout=5)
configure("sensor", timeout=5, retries=2)

## Po co keyword-only?

Porównaj:

```python
configure("sensor", 5, 2)
```

z:

```python
configure(
    "sensor",
    timeout=5,
    retries=2
)
```

Drugi zapis jest dłuższy, ale dużo bardziej czytelny.

Keyword-only jest więc narzędziem **projektowania interfejsu funkcji**, a nie tylko ciekawostką składniową.

## Argumenty positional-only

Python pozwala również wymusić przekazywanie argumentów wyłącznie pozycyjnie:

```python
def function(a, b, /):
    ...
```

Znak `/` oddziela parametry positional-only.

Ten zapis często spotkasz w dokumentacji funkcji wbudowanych i bibliotek.

In [ ]:
def scale(value, factor, /):
    return value * factor

print(scale(10, 2))

# Niepoprawne:
# scale(value=10, factor=2)

## Pełna mapa parametrów

Możemy spotkać zapis:

```python
def function(a, /, b, *, c):
    ...
```

Interpretacja:

```text
a       → tylko pozycyjnie
b       → pozycyjnie lub nazwane
c       → tylko nazwane
```

Na tym etapie ważniejsze jest umieć **czytać** taką sygnaturę niż projektować bardzo złożone API.

In [ ]:
def example(a, /, b, *, c):
    print(a, b, c)

example(1, 2, c=3)
example(1, b=2, c=3)

## Funkcja jest obiektem

W Wykładzie 2 przyjęliśmy model:

```text
nazwa → obiekt
```

Funkcja nie jest wyjątkiem.

Po wykonaniu:

```python
def say_hello():
    print("Hello")
```

nazwa `say_hello` wskazuje na **obiekt funkcji**.

In [ ]:
def say_hello():
    print("Hello")

print(say_hello)
print(type(say_hello))
print(callable(say_hello))

## Jedna funkcja może mieć więcej niż jedną nazwę

Skoro funkcja jest obiektem, możemy związać z nią kolejną nazwę:

In [ ]:
def say_hello():
    print("Hello")

function = say_hello

say_hello()
function()

print(say_hello is function)

## Uwaga na nawiasy

Porównaj:

```python
function = say_hello
```

z:

```python
function = say_hello()
```

Pierwszy zapis przypisuje **obiekt funkcji**.

Drugi najpierw **wywołuje funkcję**, a następnie przypisuje jej wynik.

In [ ]:
def get_number():
    return 42

function = get_number
result = get_number()

print(function)
print(result)

## Funkcję można przekazać jako argument

Jeżeli funkcja jest obiektem, możemy przekazać ją do innej funkcji.

In [ ]:
def square(value):
    return value ** 2

def apply(function, value):
    return function(value)

print(apply(square, 5))

## Funkcja wyższego rzędu

Funkcję, która:

- przyjmuje inną funkcję jako argument lub
- zwraca funkcję

nazywamy **funkcją wyższego rzędu**.

Jest to ważna cecha Pythona i podstawa wielu mechanizmów, które zobaczymy później.

## Przykład praktyczny: różne sposoby przetwarzania

Ta sama funkcja może zastosować różne operacje do danych:

In [ ]:
def double(value):
    return value * 2

def square(value):
    return value ** 2

def transform(values, operation):
    result = []

    for value in values:
        result.append(operation(value))

    return result

numbers = [1, 2, 3, 4]

print(transform(numbers, double))
print(transform(numbers, square))

## `lambda` — mała funkcja bez osobnej definicji

Wyrażenie:

```python
lambda x: x ** 2
```

tworzy obiekt funkcji odpowiadający mniej więcej:

```python
def square(x):
    return x ** 2
```

`lambda` może zawierać tylko pojedyncze wyrażenie.

In [ ]:
square = lambda x: x ** 2

print(square(5))
print(type(square))

## Kiedy `lambda` ma sens?

Najczęściej wtedy, gdy potrzebujemy krótkiej funkcji **w miejscu użycia**.

Przykład:

In [ ]:
students = [
    ("Anna", 4.5),
    ("Jan", 3.5),
    ("Maria", 5.0),
]

students.sort(key=lambda student: student[1])

print(students)

## Kiedy lepiej użyć `def`?

Jeżeli funkcja:

- ma być używana wielokrotnie,
- wymaga komentarza lub docstringa,
- wykonuje bardziej złożoną operację,
- ma istotne znaczenie w programie,

zwykłe `def` będzie najczęściej czytelniejsze.

`lambda` nie jest „lepszą” ani „krótszą wersją” każdej funkcji.

## Funkcja wewnątrz funkcji

Python pozwala definiować funkcje zagnieżdżone.

In [ ]:
def outer():
    def inner():
        print("Funkcja wewnętrzna")

    inner()

outer()

## Zakres nazw — przypomnienie

Funkcja wewnętrzna ma dostęp do nazw funkcji zewnętrznej.

In [ ]:
def greet(name):
    prefix = "Witaj"

    def create_message():
        return f"{prefix}, {name}!"

    return create_message()

print(greet("Anna"))

## Funkcja może zwrócić inną funkcję

To już bardziej interesujący przypadek:

In [ ]:
def create_multiplier(multiplier):

    def multiply(value):
        return value * multiplier

    return multiply

double = create_multiplier(2)
triple = create_multiplier(3)

print(double(10))
print(triple(10))

## Closure — funkcja „pamięta” otoczenie

Po zakończeniu:

```python
create_multiplier(2)
```

lokalna funkcja `multiply` nadal korzysta z wartości `multiplier = 2`.

W uproszczeniu:

> closure to funkcja wraz z zapamiętanym środowiskiem potrzebnym do jej działania.

Nie musimy jeszcze znać technicznych szczegółów implementacji.

## `nonlocal`

Jeżeli funkcja wewnętrzna chce **zmodyfikować** nazwę z funkcji zewnętrznej, może użyć `nonlocal`.

In [ ]:
def create_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter = create_counter()

print(counter())
print(counter())
print(counter())

## `global` a `nonlocal`

```text
global
↓
odwołuje się do nazwy na poziomie modułu

nonlocal
↓
odwołuje się do nazwy w otaczającej funkcji
```

`nonlocal` pojawia się głównie przy funkcjach zagnieżdżonych i closures.

## Po co nam to wszystko?

Mechanizmy z dzisiejszego wykładu prowadzą do ważnych elementów Pythona:

```text
funkcja jako obiekt
        ↓
funkcja jako argument
        ↓
funkcja zwracająca funkcję
        ↓
closure
        ↓
dekoratory
```

Dekoratory omówimy osobno, gdy ten model będzie już utrwalony.

## Ćwiczenie integrujące

Zbuduj funkcję:

```python
process_measurements(values, operation, *, ignore_negative=True)
```

Wymagania:

1. `values` — kolekcja pomiarów,
2. `operation` — funkcja wykonywana dla każdego pomiaru,
3. `ignore_negative` — argument tylko nazwany,
4. jeśli `ignore_negative=True`, pomijaj wartości ujemne,
5. zwróć listę wyników.

Przetestuj ją z:

```python
lambda x: x * 2
lambda x: x ** 2
```

In [ ]:
def process_measurements(values, operation, *, ignore_negative=True):
    result = []

    for value in values:
        if ignore_negative and value < 0:
            continue

        result.append(operation(value))

    return result

measurements = [2, -1, 3, 4]

print(process_measurements(
    measurements,
    lambda x: x * 2
))

print(process_measurements(
    measurements,
    lambda x: x ** 2,
    ignore_negative=False
))

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- `*` rozwija argumenty pozycyjne, a `*args` je zbiera,
- `**` rozwija argumenty nazwane, a `**kwargs` je zbiera,
- keyword-only może poprawiać czytelność API,
- positional-only często spotkamy w dokumentacji,
- funkcja w Pythonie jest obiektem,
- funkcję można przypisać do nazwy, przekazać i zwrócić,
- `lambda` tworzy małą funkcję będącą wyrażeniem,
- funkcja zagnieżdżona może korzystać z otaczającego zakresu,
- closure może „pamiętać” wartości z miejsca utworzenia,
- `nonlocal` pozwala modyfikować nazwę z otaczającej funkcji.

## Następny wykład

# Moduły, pakiety i organizacja programu

Skoro umiemy już dzielić program na funkcje, kolejne pytanie brzmi:

> Jak podzielić większy program pomiędzy wiele plików i pakietów?

Poznamy:

- moduły,
- `import`,
- `__name__`,
- pakiety,
- bibliotekę standardową,
- środowiska wirtualne,
- instalowanie zależności.